# Phase 9: Agentic RAG System Evaluation

**作成日**: 2026-02-03  
**プロジェクト**: experiments-local-llm  
**目的**: Agentic RAGシステムの評価と既存システムとの比較

---

## 評価対象システム

| システム | 説明 | 実装状況 |
|---------|------|----------|
| **StructuredRAG** | Phase 6で実装（Phase 8で89.1%達成） | 完了 |
| **GraphRAG** | Phase 8で実装（76.7%） | 完了 |
| **Adaptive RAG** | Phase 8で実装（86.1%） | 完了 |
| **Agentic RAG** ★ | Phase 9で実装（LangGraphベース） | 評価対象 |

## テストケース

- 既存テストケース（Phase 5-6）: 55件
- GraphRAG向けテストケース（Phase 8）: 35件
- Agentic RAG向けテストケース（Phase 9）: 15件
- **合計: 105件**

## 期待される結果

- 複数ステップ推論タスクでの精度向上
- ツール選択の適切性
- 全体的な精度とベースライン比較

## 1. 環境セットアップ

Google Colab環境でTransformersとLangGraphをセットアップします。

**注**: Phase 8と同様に、HuggingFace Transformersを使用します（Ollamaは不要）。

In [ ]:
# Google Colab環境チェック
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # プロジェクトパスを設定
    PROJECT_PATH = '/content/drive/MyDrive/experiments-local-llm'
    sys.path.insert(0, f'{PROJECT_PATH}/src')
    
    # 必要なパッケージをインストール
    !pip install -q chromadb sentence-transformers networkx
    !pip install -q transformers accelerate bitsandbytes
    !pip install -q langchain langchain-core langchain-community langchain-chroma langgraph
    !pip install -q matplotlib seaborn pandas numpy tqdm
else:
    PROJECT_PATH = '..'
    sys.path.insert(0, f'{PROJECT_PATH}/src')

print(f"✓ Project path: {PROJECT_PATH}")
print(f"✓ Running in Colab: {IN_COLAB}")

In [ ]:
import json
import time
import torch
from datetime import datetime
from typing import List, Dict, Any
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from sentence_transformers import SentenceTransformer
from langchain_community.llms import HuggingFacePipeline
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ CUDA device: {torch.cuda.get_device_name(0)}")

## 2. LLMの初期化

Qwen2.5-7B-Instructモデルを4bit量子化でロードします（Phase 8と同様）。

In [ ]:
# モデル名
model_name = "Qwen/Qwen2.5-7B-Instruct"

print(f"Loading {model_name}...")

# 4bit量子化設定
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# トークナイザーロード
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)
print("✓ Tokenizer loaded")

# モデルロード
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
print("✓ Model loaded (4-bit quantized)")

# パイプライン作成
text_generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.0,
    do_sample=False,
    return_full_text=False
)

# LangChain用LLMラッパー
llm = HuggingFacePipeline(pipeline=text_generation_pipeline)
print("✓ LLM pipeline ready")

## 3. POIデータの読み込み

In [ ]:
# POIデータの読み込み
from geo_utils import enrich_all_pois

poi_file = f"{PROJECT_PATH}/poi_documents.json"
print(f"Loading POI data from {poi_file}...")

with open(poi_file, "r", encoding="utf-8") as f:
    raw_pois = json.load(f)

# メタデータをフラット化
flat_pois = []
for poi in raw_pois:
    if "metadata" in poi:
        flat_poi = poi["metadata"].copy()
        flat_pois.append(flat_poi)
    else:
        flat_pois.append(poi)

# 空間情報付加
pois = enrich_all_pois(flat_pois)

print(f"✓ Loaded {len(pois)} POIs with spatial info")
print(f"  Sample POI: {pois[0].get('name')} ({pois[0].get('category')})")

## 4. テストケースの読み込み

In [ ]:
# テストケースのインポート
from test_cases_v2 import TEST_CASES_V2 as STRUCTURED_TEST_CASES
from test_cases_graphrag import GRAPHRAG_TEST_CASES as ALL_GRAPHRAG_TEST_CASES
from test_cases_agentic import ALL_AGENTIC_TEST_CASES

print("Test Cases Loaded:")
print(f"  Structured (Phase 5-6): {len(STRUCTURED_TEST_CASES)} cases")
print(f"  GraphRAG (Phase 8): {len(ALL_GRAPHRAG_TEST_CASES)} cases")
print(f"  Agentic (Phase 9): {len(ALL_AGENTIC_TEST_CASES)} cases")
print(f"  Total: {len(STRUCTURED_TEST_CASES) + len(ALL_GRAPHRAG_TEST_CASES) + len(ALL_AGENTIC_TEST_CASES)} cases")

# テストケースの統合
ALL_TEST_CASES = STRUCTURED_TEST_CASES + ALL_GRAPHRAG_TEST_CASES + ALL_AGENTIC_TEST_CASES

## 5. システムの初期化

AgenticRAG と StructuredRAG の両方を初期化します。

**✓ AgenticRAG**: HuggingFace Transformers対応完了（LangGraph + ReAct）

In [ ]:
from structured_rag_system import StructuredRAGSystem
from agentic_rag_system import AgenticRAGSystem
from agent_tools import set_global_pois
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

# グローバルPOI設定（ツール用）
set_global_pois(pois)
print("✓ Global POIs set for tools")

# ベクトルストア作成（StructuredRAG用）
print("\nCreating vectorstore...")
documents = [
    Document(
        page_content=f"{poi.get('name', '')} {poi.get('category', '')} {poi.get('description', '')}",
        metadata=poi
    )
    for poi in pois
]

# 埋め込みモデル
embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)

# Chromaベクトルストア作成
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="poi_collection"
)
print(f"✓ Vectorstore created with {len(documents)} documents")

# StructuredRAGシステム初期化
print("\nInitializing Structured RAG system...")
structured_system = StructuredRAGSystem(
    model=model,
    tokenizer=tokenizer,
    vectorstore=vectorstore,
    all_pois=pois,
    debug=False
)
print("✓ Structured RAG initialized")

# Agentic RAGシステム初期化（事前ロードされたモデルを使用）
print("\nInitializing Agentic RAG system...")
agentic_system = AgenticRAGSystem(
    model=model,
    tokenizer=tokenizer,
    model_name=model_name,
    verbose=False,  # 評価時はログを抑制
    max_iterations=5  # 最大5イテレーション
)
print("✓ Agentic RAG initialized")

print("\n✓ All systems ready for evaluation")

## 6. 評価関数の定義

In [ ]:
def evaluate_keyword_hit_rate(answer: str, expected_keywords: List[str]) -> float:
    """キーワードヒット率を計算"""
    if not expected_keywords:
        return 1.0
    # None または空文字列のチェック
    if not answer:
        return 0.0
    answer_lower = answer.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in answer_lower)
    return hits / len(expected_keywords)

def evaluate_single_case(system, test_case, system_name: str) -> Dict[str, Any]:
    """単一テストケースの評価
    
    注: 異なるテストケースクラスの属性名に対応:
    - TestCaseV2, AgenticTestCase: .prompt
    - GraphRAGTestCase: .question
    """
    try:
        # 質問文を取得（promptまたはquestion属性）
        question = getattr(test_case, 'prompt', None) or getattr(test_case, 'question', '')
        
        start_time = time.time()
        
        # システム実行
        result = system.query(question)
        answer = result.get('answer', '') or ''  # None対策
        execution_time = time.time() - start_time
        
        # キーワードヒット率
        keyword_hit_rate = evaluate_keyword_hit_rate(answer, test_case.expected_keywords)
        
        # 成功判定（50%以上）
        success = keyword_hit_rate >= 0.5
        
        return {
            'test_id': test_case.id,
            'category': getattr(test_case, 'subcategory', test_case.category),
            'question': question,
            'answer': answer,
            'execution_time': execution_time,
            'keyword_hit_rate': keyword_hit_rate,
            'success': success,
            'error': None
        }
    except Exception as e:
        # エラー時も質問文を取得
        question = getattr(test_case, 'prompt', None) or getattr(test_case, 'question', '')
        
        return {
            'test_id': test_case.id,
            'category': getattr(test_case, 'subcategory', test_case.category),
            'question': question,
            'answer': '',
            'execution_time': 0,
            'keyword_hit_rate': 0.0,
            'success': False,
            'error': str(e)
        }

print("✓ Evaluation functions defined (with multi-format test case support)")

## 7. 評価実行

**注意**: 全105ケースの評価には時間がかかります（30-60分程度）。
クイックテストの場合は、`quick_test = True`に設定してください。

In [ ]:
# クイックテスト設定（最初の10ケースのみ）
quick_test = True  # Trueでクイックテスト、Falseで全ケース評価

if quick_test:
    test_cases = ALL_TEST_CASES[:10]
    print("⚠ Quick test mode: evaluating first 10 cases only")
else:
    test_cases = ALL_TEST_CASES
    print(f"Running full evaluation: {len(test_cases)} cases")

print("\n" + "="*60)
print("Starting Evaluation")
print("="*60)

# Structured RAG評価
print("\nEvaluating Structured RAG (baseline)...")
structured_results = []
for tc in tqdm(test_cases, desc="Structured RAG"):
    result = evaluate_single_case(structured_system, tc, "Structured")
    structured_results.append(result)

# Agentic RAG評価
print("\nEvaluating Agentic RAG...")
agentic_results = []
for tc in tqdm(test_cases, desc="Agentic RAG"):
    result = evaluate_single_case(agentic_system, tc, "Agentic")
    agentic_results.append(result)

print("\n✓ Evaluation completed")

## 8. 結果分析

In [ ]:
import pandas as pd
import numpy as np

# データフレーム作成
df_structured = pd.DataFrame(structured_results)
df_agentic = pd.DataFrame(agentic_results)

# 全体スコア計算
def calculate_metrics(df):
    return {
        'success_rate': (df['success'].sum() / len(df)) * 100,
        'avg_hit_rate': df['keyword_hit_rate'].mean() * 100,
        'avg_execution_time': df['execution_time'].mean(),
        'std_execution_time': df['execution_time'].std(),
        'error_count': df['error'].notna().sum()
    }

structured_metrics = calculate_metrics(df_structured)
agentic_metrics = calculate_metrics(df_agentic)

print("="*60)
print("Overall Results")
print("="*60)

print(f"\n{'Metric':<25} {'Structured RAG':<18} {'Agentic RAG':<15}")
print("-"*60)
print(f"{'Success Rate':<25} {structured_metrics['success_rate']:<18.1f}% {agentic_metrics['success_rate']:<15.1f}%")
print(f"{'Avg Hit Rate':<25} {structured_metrics['avg_hit_rate']:<18.1f}% {agentic_metrics['avg_hit_rate']:<15.1f}%")
print(f"{'Avg Execution Time':<25} {structured_metrics['avg_execution_time']:<18.2f}s {agentic_metrics['avg_execution_time']:<15.2f}s")
print(f"{'Error Count':<25} {structured_metrics['error_count']:<18} {agentic_metrics['error_count']:<15}")

# カテゴリ別スコア
print("\n" + "="*60)
print("Results by Category")
print("="*60)

category_results = []
for category in df_structured['category'].unique():
    structured_cat = df_structured[df_structured['category'] == category]
    agentic_cat = df_agentic[df_agentic['category'] == category]
    
    category_results.append({
        'category': category,
        'count': len(structured_cat),
        'structured_rate': (structured_cat['success'].sum() / len(structured_cat)) * 100,
        'agentic_rate': (agentic_cat['success'].sum() / len(agentic_cat)) * 100
    })

df_category = pd.DataFrame(category_results)
df_category['improvement'] = df_category['agentic_rate'] - df_category['structured_rate']
df_category = df_category.sort_values('improvement', ascending=False)

print(f"\n{'Category':<25} {'Count':<8} {'Structured':<12} {'Agentic':<12} {'Delta'}")
print("-"*70)
for _, row in df_category.iterrows():
    delta_str = f"{row['improvement']:+.1f}%" if row['improvement'] != 0 else "0.0%"
    print(f"{row['category']:<25} {row['count']:<8} {row['structured_rate']:<12.1f}% {row['agentic_rate']:<12.1f}% {delta_str}")

## 9. 可視化

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

# カテゴリ別成功率の比較
fig, ax = plt.subplots(figsize=(14, 8))

x = np.arange(len(df_category))
width = 0.35

bars1 = ax.barh(x - width/2, df_category['structured_rate'], width, label='Structured RAG', color='#3498db', alpha=0.7)
bars2 = ax.barh(x + width/2, df_category['agentic_rate'], width, label='Agentic RAG', color='#e74c3c', alpha=0.7)

ax.set_xlabel('Success Rate (%)', fontsize=12)
ax.set_ylabel('Category', fontsize=12)
ax.set_title('Success Rate by Category: Structured vs Agentic RAG', fontsize=14, fontweight='bold')
ax.set_yticks(x)
ax.set_yticklabels(df_category['category'])
ax.legend(fontsize=11)
ax.set_xlim([0, 110])

# 値のラベル追加
for i, (s, a) in enumerate(zip(df_category['structured_rate'], df_category['agentic_rate'])):
    ax.text(s + 2, i - width/2, f'{s:.0f}%', va='center', fontsize=8)
    ax.text(a + 2, i + width/2, f'{a:.0f}%', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('phase9_comparison_results.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualization saved")

## 10. 結果の保存

In [ ]:
# タイムスタンプ付きファイル名
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# numpy型をPython標準型に変換する関数
def convert_to_serializable(obj):
    """numpy型をJSON serializableな型に変換"""
    if isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {key: convert_to_serializable(value) for key, value in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_serializable(item) for item in obj]
    else:
        return obj

# メトリクスを変換
structured_metrics_converted = convert_to_serializable(structured_metrics)
agentic_metrics_converted = convert_to_serializable(agentic_metrics)
category_results_converted = convert_to_serializable(df_category.to_dict('records'))

# 詳細結果の保存
results_data = {
    'timestamp': timestamp,
    'test_count': len(test_cases),
    'quick_test': quick_test,
    'structured_rag': {
        'metrics': structured_metrics_converted,
        'results': convert_to_serializable(structured_results)
    },
    'agentic_rag': {
        'metrics': agentic_metrics_converted,
        'results': convert_to_serializable(agentic_results)
    },
    'category_results': category_results_converted
}

output_file = f'{PROJECT_PATH}/results/phase9_evaluation_{timestamp}.json'
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(results_data, f, ensure_ascii=False, indent=2)

print(f"✓ Results saved to {output_file}")

# サマリーレポート
summary_file = f'{PROJECT_PATH}/results/phase9_summary_{timestamp}.txt'
with open(summary_file, 'w', encoding='utf-8') as f:
    f.write("Phase 9: Agentic RAG Evaluation Summary\n")
    f.write("="*60 + "\n\n")
    f.write(f"Evaluation Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Test Cases: {len(test_cases)}\n")
    f.write(f"Quick Test Mode: {quick_test}\n\n")
    
    f.write("Overall Results:\n")
    f.write("-"*60 + "\n")
    f.write(f"Structured RAG Success Rate: {structured_metrics['success_rate']:.1f}%\n")
    f.write(f"Agentic RAG Success Rate: {agentic_metrics['success_rate']:.1f}%\n")
    f.write(f"Improvement: {agentic_metrics['success_rate'] - structured_metrics['success_rate']:+.1f}%\n\n")
    
    f.write(f"Structured RAG Avg Execution Time: {structured_metrics['avg_execution_time']:.2f}s\n")
    f.write(f"Agentic RAG Avg Execution Time: {agentic_metrics['avg_execution_time']:.2f}s\n\n")
    
    f.write("\nCategory-wise Results:\n")
    f.write("-"*70 + "\n")
    f.write(f"{'Category':<25} {'Structured':<12} {'Agentic':<12} {'Delta'}\n")
    f.write("-"*70 + "\n")
    for _, row in df_category.iterrows():
        delta_str = f"{row['improvement']:+.1f}%" if row['improvement'] != 0 else "0.0%"
        f.write(f"{row['category']:<25} {row['structured_rate']:<12.1f}% {row['agentic_rate']:<12.1f}% {delta_str}\n")

print(f"✓ Summary saved to {summary_file}")
print("\n✓ Evaluation complete!")

## 11. 結論と次のステップ

### ✅ 実装完了

Phase 9: Agentic RAGシステムのHuggingFace Transformers統合が**完了**しました。

**主要な成果**:
1. **LangGraph + HuggingFace統合**: ChatOllamaからTransformersへの完全移行
2. **ReActスタイル推論**: ツール呼び出しをプロンプトベースで実装
3. **16個のツール**: 空間計算、集計、グラフトラバーサルツールの完全実装
4. **Google Colab対応**: 4bit量子化で効率的に実行可能

### 評価結果の分析

**期待される結果**:
- 複雑な多段階推論タスクでAgenticRAGが優位性を示す
- 単純なクエリではStructuredRAGが効率的
- カテゴリ別に見ると、conditional_reasoning/iterative_refinementで大きな改善

### アーキテクチャの特徴

**Agentic RAG**:
- エージェントループ（最大5イテレーション）
- ツール選択と実行の動的判断
- 中間結果を踏まえた段階的推論

**Structured RAG**:
- 単一パスの質問分析
- ルールベースのコンテキスト構築
- 高速だが柔軟性に欠ける

### 今後の改善方向

1. **プロンプトエンジニアリング**: ReActプロンプトの最適化
2. **ツール選択戦略**: より賢いツール選択アルゴリズム
3. **Self-correction機構**: エラー検出と修正のループ
4. **Phase 10: 全国展開**: PostGIS/Supabaseへの移行